In [ ]:
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
import os

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
import tensorflow as tf

# Ultra-safe fallback settings for stability first.
IMG_SIZE = (300,300)
BATCH_SIZE = 32

data_dir = '/Users/smit/Desktop/DATA SCIENCE/Deep Learning/Flower dataset'

print('IMG_SIZE =', IMG_SIZE, '| BATCH_SIZE =', BATCH_SIZE)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
seed = 1337

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='training',
    seed=seed,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset='validation',
    seed=seed,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
num_classes = len(class_names)
class_indices = {name: idx for idx, name in enumerate(class_names)}

# Keep preprocessing minimal for stability.
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE).prefetch(1)
val_ds = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTOTUNE).prefetch(1)

print('num_classes =', num_classes)
print('class_indices =', class_indices)


In [ ]:
class_indices, num_classes, class_names

In [ ]:
import matplotlib.pyplot as plt

nrows = 3
ncols = 5
fig = plt.gcf()
fig.set_size_inches(ncols * 4, nrows * 4)

images, labels_batch = next(iter(train_ds))

for i in range(0, nrows * ncols):
    ax = plt.subplot(nrows, ncols, i + 1)
    ax.axis('off')
    plt.imshow(images[i].numpy())
    class_idx = int(labels_batch[i].numpy())
    plt.title(class_names[class_idx])

plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import numpy as np
import tensorflow as tf

In [ ]:
def get_model(num_classes: int):
    model = Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        Conv2D(16, (3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(32, (3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        tf.keras.layers.GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax'),
    ])
    return model

In [ ]:
model = get_model(num_classes)

In [ ]:
model.summary()

In [ ]:
model.layers

In [ ]:
weights, biases = model.layers[0].get_weights()

In [ ]:
len(biases), len(weights)

In [ ]:
model.layers[1].get_weights()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
    ModelCheckpoint('Flowers_best.keras', monitor='val_accuracy', save_best_only=True),
]
# 2) Short stable run first.
history = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
history.history

In [ ]:
f5fix, ax = plt.subplots(2, 1)
ax[0].plot(history.history['loss'], color='b', label='Training Loss')
ax[0].plot(history.history['val_loss'], color='r', label='Validation Loss')
ax[0].legend()
ax[1].plot(history.history['accuracy'], color='b', label='Training accuracy')
ax[1].plot(history.history['val_accuracy'], color='r', label='Validationaccuracy')
ax[1].legend()

In [ ]:
model.save('Flowers.h5')

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

In [ ]:
# Load the best checkpoint (saved during training)
model_load = load_model('Flowers.h5')

In [ ]:
image_path = '/Users/smit/Desktop/DATA SCIENCE/Deep Learning/Flower dataset/rose/160954292_6c2b4fda65_n.jpg'

img = image.load_img(image_path, target_size=IMG_SIZE)
img = image.img_to_array(img)
img = np.expand_dims(img, axis=0)
img = img / 255.0

prediction = model_load.predict(img)
predicted_class = int(np.argmax(prediction, axis=1)[0])
confidence = float(np.max(prediction))
print('raw prediction:', prediction)
print('predicted_class:', predicted_class, 'confidence:', confidence)

In [ ]:
labels = {v: k for k, v in class_indices.items()}
print('The image is of a', labels[predicted_class])